# Step 02. Dijkstra 알고리즘

> **출처**: PythonRobotics (https://github.com/AtsushiSakai/PythonRobotics)
> Original author: Atsushi Sakai (@Atsushi_twi) / License: MIT
>
> **교육 주관**: Hanyang University, Automotive Intelligence Laboratory (AILAB)

연속 공간을 격자로 이산화한 뒤, **비용이 가장 낮은 노드부터 확장**하는 Dijkstra로 최단 경로를 찾습니다.

> **실습 안내**: 각 `# TODO` 아래 줄의 `=` 뒤(우변)를 채운 뒤, 위에서부터 차례로 실행하세요.
> - TODO 1 (섹션 9): expand_once - 이웃 노드 생성
> - TODO 2 (섹션 10): reconstruct_path - parent 체인 역추적
> - TODO 3 (섹션 11): dijkstra - 목표 도달 처리 + 이웃 노드 생성

## 학습 목표

1. Grid Map / 장애물 팽창 이해
2. Open Set(탐색 경계) / Closed Set(방문 완료) 개념
3. **'한 번의 확장'** 을 직접 보고, 전체 탐색이 그 반복임을 이해
4. 경로 역추적 + 탐색 과정 애니메이션

In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2
import sys, os, math, heapq, time
from collections import namedtuple
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import HTML

from utils.grid_map import make_simple_map, build_obstacle_map
from utils.visualization import plot_grid_map, plot_search_result
from utils.grid_search import (calc_grid_index, calc_grid_position,
                               get_motion_model, is_valid)
from utils.search_viz import animate_search

print('라이브러리 로드 완료')

## 1. Grid Map 생성

Dijkstra는 **그래프** 위에서 동작합니다. 연속 공간을 격자로 바꿔 각 셀을 노드로 삼습니다.
출발(-5,-5)에서 목표(50,50)까지 벽을 피해 경로를 찾습니다.

In [ ]:
ox, oy, sx, sy, gx, gy = make_simple_map()
print(f'장애물 {len(ox)}개,  출발 ({sx},{sy})  ->  목표 ({gx},{gy})')

fig, ax = plt.subplots(figsize=(6, 6))
plot_grid_map(ox, oy, sx, sy, gx, gy, ax=ax)
ax.grid(False)
ax.set_title('Simple Grid Map')
plt.tight_layout(); plt.show()

## 2. 격자 파라미터

**Resolution**: 격자 한 칸 크기(작을수록 정밀·느림).
**Robot Radius**: 장애물을 이만큼 팽창시켜 차량을 점으로 단순화 (C-Space 변환).

값을 정하고, 선택한 해상도(2m) 격자를 맵 위에 그려 봅니다. 해상도가 탐색 효율(노드 수·시간·경로 길이)에 주는 영향은 **알고리즘을 직접 실행한 뒤** §14에서 비교합니다.

In [ ]:
RESOLUTION   = 2.0  # 격자 한 칸 크기 [m]
ROBOT_RADIUS = 1.0  # 장애물 팽창 반경 [m]

min_x, max_x = round(min(ox)), round(max(ox))
min_y, max_y = round(min(oy)), round(max(oy))
x_width = int((max_x - min_x) / RESOLUTION) + 1
y_width = int((max_y - min_y) / RESOLUTION) + 1

print(f'맵 범위  x[{min_x}, {max_x}]  y[{min_y}, {max_y}]')
print(f'해상도 {RESOLUTION} m  ->  격자 {x_width} x {y_width} = {x_width*y_width:,} cells')

# 선택한 해상도 격자를 맵 위에 그려 본다
fig, ax = plt.subplots(figsize=(6, 6))
for x in np.arange(min_x, max_x + RESOLUTION, RESOLUTION):
    ax.axvline(x, color='lightgray', lw=0.5)
for y in np.arange(min_y, max_y + RESOLUTION, RESOLUTION):
    ax.axhline(y, color='lightgray', lw=0.5)
ax.plot(ox, oy, '.k', ms=2, alpha=0.5)
ax.plot(sx, sy, 'og', ms=10); ax.plot(gx, gy, 'xb', ms=10, mew=2)
ax.set_aspect('equal'); ax.grid(False)
ax.set_title(f'{RESOLUTION}m grid on the map  ({x_width} x {y_width} cells)')
plt.tight_layout(); plt.show()

## 3. 장애물 맵 (팽창)

`build_obstacle_map`이 각 격자 셀을 **통행 가능(free)** 또는 **장애물(blocked)** 로 분류합니다.
이때 장애물을 robot_radius만큼 **팽창**시켜 차량을 점으로 단순화합니다.
robot_radius가 클수록 안전하지만 좁은 통로가 막힐 수 있습니다.

아래 두 그림은 같은 벽(빨강)에 대해 팽창 반경만 다르게 준 결과로 '벽이 두꺼워지는' 팽창 효과를 보입니다.

In [ ]:
# build_obstacle_map: 장애물을 robot_radius만큼 '팽창'시켜 통행 가능/불가 격자를 만든다
obstacle_map, min_x, min_y, max_x, max_y, x_width, y_width = \
    build_obstacle_map(ox, oy, RESOLUTION, ROBOT_RADIUS)

def blocked_cells(omap):
    bx, by = [], []
    for ix in range(len(omap)):
        for iy in range(len(omap[0])):
            if omap[ix][iy]:
                bx.append(calc_grid_position(ix, min_x, RESOLUTION))
                by.append(calc_grid_position(iy, min_y, RESOLUTION))
    return bx, by

bx1, by1 = blocked_cells(obstacle_map)
demo_map, *_ = build_obstacle_map(ox, oy, RESOLUTION, 4.0)   # robot_radius를 키운 비교용 맵
bx4, by4 = blocked_cells(demo_map)

# 출발/목표(연속 좌표)를 가장 가까운 격자 셀 중심으로 변환
sx_c = calc_grid_position(calc_grid_index(sx, min_x, RESOLUTION), min_x, RESOLUTION)
sy_c = calc_grid_position(calc_grid_index(sy, min_y, RESOLUTION), min_y, RESOLUTION)
gx_c = calc_grid_position(calc_grid_index(gx, min_x, RESOLUTION), min_x, RESOLUTION)
gy_c = calc_grid_position(calc_grid_index(gy, min_y, RESOLUTION), min_y, RESOLUTION)

fig, axes = plt.subplots(1, 2, figsize=(13, 6.5))
for ax in axes:
    ax.plot(ox, oy, '.', color='red', ms=2, alpha=0.5)
    ax.plot(sx_c, sy_c, 'og', ms=11, zorder=5)
    ax.plot(gx_c, gy_c, 'xb', ms=11, mew=2.5, zorder=5)
    ax.set_aspect('equal'); ax.grid(True, alpha=0.2)
    ax.set_xlim(min_x - 3, max_x + 3); ax.set_ylim(min_y - 3, max_y + 3)
    
axes[0].plot(bx1, by1, 's', color='dimgray', ms=4, alpha=0.85)
axes[0].set_title(f'inflated robot_radius={ROBOT_RADIUS}m')
axes[1].plot(bx4, by4, 's', color='dimgray', ms=4, alpha=0.85)
axes[1].set_title('inflated robot_radius=4m')
plt.suptitle('Obstacle inflation : red = raw wall, gray = blocked cells', fontsize=13)
plt.tight_layout(); plt.subplots_adjust(top=0.90); plt.show()
print(f'robot_radius={ROBOT_RADIUS}m -> 통행 가능 {x_width*y_width - len(bx1)}개 / 장애물(팽창) {len(bx1)}개')

## 4. Node - 상태 표현

노드 하나 = 탐색 상태 하나. 저장 정보:

| 필드 | 의미 |
|------|------|
| `x, y` | 격자 인덱스 |
| `cost` | 출발부터 누적 비용 g(n) |
| `parent_index` | 직전 노드 키 (역추적용), 시작은 -1 |

In [ ]:
class Node:
    # 격자 탐색의 기본 단위. (x, y)는 실제 좌표가 아닌 격자 인덱스.
    def __init__(self, x, y, cost, parent_index):
        self.x = x
        self.y = y
        self.cost = cost                   # 출발점부터 누적 비용 g(n)
        self.parent_index = parent_index   # 직전 노드 키 (ix, iy), 시작은 -1
    def __lt__(self, other):
        return self.cost < other.cost      # heapq 비교용

# 실제 출발 지점을 노드로 표현 (좌표 -> 격자 인덱스)
example_node = Node(calc_grid_index(sx, min_x, RESOLUTION),
                    calc_grid_index(sy, min_y, RESOLUTION), 0.0, parent_index=-1)
print(f'출발 ({sx},{sy}) -> 격자 인덱스 ({example_node.x},{example_node.y}), '
      f'cost={example_node.cost}, parent={example_node.parent_index}')

## 5. 좌표 변환 헬퍼

```
calc_grid_index    : 좌표[m]    -> 격자 인덱스
calc_grid_position : 격자 인덱스 -> 좌표[m]
```

Open/Closed Set의 **dict 키로는 격자 인덱스 `(ix, iy)` 튜플을 그대로** 사용합니다.
(참고: 실제 구현에서는 속도를 위해 2D 인덱스를 1D 정수로 평탄화하기도 합니다.)

In [ ]:
# 출발 좌표 하나를 좌표 -> 인덱스 -> (다시) 좌표 로 왕복시켜 변환 함수를 확인
ix = calc_grid_index(sx, min_x, RESOLUTION)       # 좌표[m] -> 격자 인덱스
iy = calc_grid_index(sy, min_y, RESOLUTION)
back_x = calc_grid_position(ix, min_x, RESOLUTION)  # 격자 인덱스 -> 좌표[m]
back_y = calc_grid_position(iy, min_y, RESOLUTION)
print(f'출발 좌표 ({sx}, {sy}) m')
print(f'  -> calc_grid_index    -> 격자 인덱스 ({ix}, {iy})')
print(f'  -> calc_grid_position -> 좌표 ({back_x}, {back_y}) m   (해상도 {RESOLUTION}m 격자에 스냅됨)')
print(f'  -> dict 키            -> (ix, iy) = ({ix}, {iy})')

## 6. 이동 모델 (Motion Model)

그래프의 **엣지**. 8방향 이동, 비용은 유클리드 거리 (직선 1.0, 대각선 $\sqrt{2}$).
$\sqrt{2}$를 써야 등방성(거리 왜곡 없음)이 보장됩니다.

> 모든 이동 비용이 **0 이상** 임에 주목하세요. 이것이 Dijkstra가 최적 경로를 보장하기 위한 전제(음수 가중치 없음)입니다.

In [ ]:
motion = get_motion_model()

def dir_label(dx, dy):
    # (dx, dy) 부호로 방향 이름을 만든다
    vertical   = {1: 'U', 0: '', -1: 'D'}[dy]
    horizontal = {1: 'R', 0: '', -1: 'L'}[dx]
    return vertical + horizontal

DIRS = [dir_label(dx, dy) for dx, dy, _ in motion]

print('8방향 이동 모델 (직선 1.0, 대각선 sqrt(2))')
for label, (dx, dy, cost) in zip(DIRS, motion):
    move_kind = 'straight' if cost == 1.0 else 'diagonal'
    print(f'  {label:<3} dx={dx:>2} dy={dy:>2} cost={cost:.3f} ({move_kind})')

In [ ]:
# 8방향을 한 그림에 (직선=파랑, 대각선=주황)
fig, ax = plt.subplots(figsize=(6, 6))
ax.set_xlim(-2.4, 2.4); ax.set_ylim(-2.4, 2.4); ax.set_aspect('equal')
ax.axhline(0, color='k', lw=0.6); ax.axvline(0, color='k', lw=0.6)
ax.grid(True, alpha=0.3); ax.set_xticks([-1, 0, 1]); ax.set_yticks([-1, 0, 1])

for (dx, dy, cost), label in zip(motion, DIRS):
    color = 'royalblue' if cost == 1.0 else 'darkorange'
    ax.annotate('', xy=(dx * 1.5, dy * 1.5), xytext=(dx * 0.35, dy * 0.35), arrowprops=dict(arrowstyle='->', color=color, lw=2.2))
    ax.text(dx * 1.95, dy * 1.95, f'{label}\n{cost:.2f}', ha='center', va='center', color=color, fontsize=9, fontweight='bold')
    
ax.text(0, 0, 'current', ha='center', va='center', fontsize=9, fontweight='bold', bbox=dict(boxstyle='round,pad=0.3', facecolor='gold', edgecolor='none'))
ax.set_title('8-Direction Motion Model\n(blue = straight 1.0, orange = diagonal sqrt2)', fontsize=11)
plt.tight_layout(); plt.show()

## 7. is_valid - 통행 가능한 이웃인가?

이웃 노드를 만들 때마다 **격자 범위 안 + 장애물 아님**을 확인해야 합니다.
`is_valid`가 이 두 조건을 검사합니다. 출발 노드의 8이웃에 적용해 봅니다.

In [ ]:
# 한 노드의 8방향 이웃이 통행 가능한지 검사 (범위 밖 / 장애물이면 불가)
start = Node(calc_grid_index(sx, min_x, RESOLUTION),
             calc_grid_index(sy, min_y, RESOLUTION), 0.0, -1)

print('출발 노드의 8이웃 검사:')
fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(ox, oy, '.k', ms=2, alpha=0.4)
ax.plot(sx, sy, 'og', ms=12, zorder=5)

for label, (dx, dy, cost) in zip(DIRS, motion):
    neighbor = Node(start.x + dx, start.y + dy, cost, -1)
    is_passable = is_valid(neighbor, obstacle_map, min_x, min_y, max_x, max_y, RESOLUTION)
    px = calc_grid_position(neighbor.x, min_x, RESOLUTION)
    py = calc_grid_position(neighbor.y, min_y, RESOLUTION)
    ax.plot(px, py, 'o', color='limegreen' if is_passable else 'red', ms=10, zorder=4)
    print(f'  {label:<3} -> {"통행 가능" if is_passable else "불가(범위밖/장애물)"}')
    
ax.set_aspect('equal'); ax.grid(True, alpha=0.2)
ax.set_title('Neighbors of Start (green=valid, red=blocked)')
plt.tight_layout(); plt.show()

## 8. Open / Closed Set 초기화

- **Open Set**: 탐색 예정 노드(= 탐색 경계). **우선순위 큐(min-heap)로 구현** - 비용이 가장 낮은 노드를 먼저 꺼냅니다. 처음엔 출발 노드 1개.
- **Closed Set**: 비용이 확정된 노드(다시 안 봄). dict로 구현하며, 키는 격자 인덱스 `(ix, iy)`.

> Open Set을 위해 별도 자료구조를 두지 않고, 우선순위 큐 하나가 그 역할을 합니다.

In [ ]:
# Open Set = 우선순위 큐(탐색 경계),  Closed Set = 비용 확정(방문 완료)
goal = Node(calc_grid_index(gx, min_x, RESOLUTION),
            calc_grid_index(gy, min_y, RESOLUTION), 0.0, -1)

priority_queue = [(0.0, start)]   # Open Set (MinHeap): (cost, node), 비용 최소부터 꺼냄
closed_set     = {}               # Closed Set: 키 = (ix, iy), 값 = Node

print(f'초기 상태:  Open Set {len(priority_queue)}개(출발),  Closed Set {len(closed_set)}개')

## 9. 한 번의 확장 (핵심)

Dijkstra의 한 스텝:

```
(1) Open Set에서 최소 비용 노드 current 를 꺼낸다
(2) current 를 Closed Set으로 옮긴다 (비용 확정)
(3) current 의 8이웃 중 (통행 가능 + 아직 Closed 아님)을 Open Set에 추가
```

아래에서 이 한 스텝을 **4번** 반복하며 두 가지를 함께 봅니다.
- **그림**: 각 셀의 숫자가 **누적 비용 g** - 비용이 낮은 셀부터 확정(회색)됩니다.
- **표**: Open Set의 `노드 | g | parent` - 각 노드의 누적 비용 g와 부모(parent)가 한 스텝마다 채워집니다.

> 같은 칸이 더 싼 경로로 Open에 다시 들어올 수 있지만, **가장 싼 것이 먼저 pop**되어 Closed로 확정되므로 이후의 더 비싼 중복은 pop될 때 `closed_set` 검사로 버려집니다.

전체 탐색(섹션 11)은 이 한 스텝을 목표에 도달할 때까지 반복할 뿐입니다.

In [ ]:
# '한 번의 확장'을 4번 반복하며 탐색 경계가 퍼지는 과정을 관찰
priority_queue = [(0.0, start)]   # Open Set
closed_set = {}

def expand_once():
    # 비용 최소 노드를 꺼내(1) Closed로 옮기고(2) 8이웃을 Open에 추가(3). 꺼낸 노드를 반환.
    while priority_queue:
        _, current = heapq.heappop(priority_queue)
        current_key = (current.x, current.y)

        if current_key in closed_set:        # 이미 확정된 노드의 중복 -> 건너뜀
            continue

        closed_set[current_key] = current
        for dx, dy, cost in motion:

            # TODO 1: 현재 노드(current)에서 한 칸 이동한 이웃 노드를 만들어 neighbor 에 담기
            # ==========================================================================
            neighbor = Node(current.x + dx, current.y + dy, current.cost + cost, current_key)
            # ==========================================================================
            
            if not is_valid(neighbor, obstacle_map, min_x, min_y, max_x, max_y, RESOLUTION):
                continue
            if (neighbor.x, neighbor.y) in closed_set:
                continue
            heapq.heappush(priority_queue, (neighbor.cost, neighbor))

        return current

def open_snapshot():
    # heap에서 (아직 Closed 아닌) 노드의 '가장 싼' 항목만 추려 표로 정리
    best = {}
    for _, node in priority_queue:
        key = (node.x, node.y)
        if key in closed_set:
            continue
        if key not in best or node.cost < best[key].cost:
            best[key] = node
    return best

to_x = lambda node: calc_grid_position(node.x, min_x, RESOLUTION)
to_y = lambda node: calc_grid_position(node.y, min_y, RESOLUTION)

fig, axes = plt.subplots(2, 2, figsize=(12, 11))
open_table = {}
for step, ax in enumerate(axes.flat, 1):       # axes.flat: 2x2 배열을 한 줄로 펴서 순회
    current = expand_once()
    open_table = open_snapshot()

    for node in closed_set.values():
        ax.plot(to_x(node), to_y(node), 's', color='gray', ms=24, alpha=0.85)
        ax.text(to_x(node), to_y(node), f'{node.cost:.1f}', ha='center', va='center', color='white', fontsize=8)

    for node in open_table.values():
        ax.plot(to_x(node), to_y(node), 's', color='deepskyblue', ms=24, alpha=0.7)
        ax.text(to_x(node), to_y(node), f'{node.cost:.1f}', ha='center', va='center', color='black', fontsize=8)
        
    ax.plot(to_x(current), to_y(current), '*', color='gold', ms=22, zorder=6, markeredgecolor='k')
    ax.set_aspect('equal'); ax.set_xlim(-12, 2); ax.set_ylim(-12, 2); ax.grid(True, alpha=0.2)
    ax.set_title(f'step {step}: pop ({current.x},{current.y}) g={current.cost:.2f}  '
                 f'| closed={len(closed_set)} open={len(open_table)}', fontsize=9)
plt.suptitle('One expansion x4  (number in cell = g, gray=Closed, skyblue=Open, star=popped)', fontsize=12)
plt.tight_layout(); plt.show()

# Open Set 상태표: 노드 | g(누적비용) | parent (비용 오름차순)
print("Open Set 상태  (노드 | g(누적비용) | parent):")
print(f"  {'node(ix,iy)':<12}{'g':>7}   parent")
for (kx, ky), node in sorted(open_table.items(), key=lambda it: it[1].cost):
    node_str = f'({kx:>2},{ky:>2})'
    parent = 'start' if node.parent_index == -1 else f'({node.parent_index[0]},{node.parent_index[1]})'
    print(f"  {node_str:<12}{node.cost:>7.2f}   {parent}")

## 10. 경로 역추적 (Path Reconstruction)

탐색이 끝나면 각 노드는 **자신의 직전 노드(parent_index)** 를 기억하고 있습니다.
목표에서 시작해 parent를 따라 거꾸로 올라가면 경로가 복원됩니다 (목표 -> ... -> 시작).

> 탐색 루프(섹션 9)는 비용과 parent만 계산할 뿐, '경로'라는 결과물은 이 역추적이 만들어 냅니다.

In [ ]:
def reconstruct_path(goal, closed_set, min_x, min_y, resolution):
    # goal의 parent_index 체인을 시작(-1)까지 따라가며 좌표를 모은 뒤 뒤집어 반환
    path_x = [calc_grid_position(goal.x, min_x, resolution)]
    path_y = [calc_grid_position(goal.y, min_y, resolution)]
    parent = goal.parent_index
    while parent != -1:                 # -1 = 시작 노드(부모 없음)
        node = closed_set[parent]
        path_x.append(calc_grid_position(node.x, min_x, resolution))
        path_y.append(calc_grid_position(node.y, min_y, resolution))

        # TODO 2: parent 를 한 칸 위(부모)로 갱신, 경로 체인 거슬러 올라가기
        # ==========================================================================
        parent = node.parent_index
        # ==========================================================================
        
    path_x.reverse()
    path_y.reverse()
    return path_x, path_y

# --- 작은 예제로 parent 체인 역추적을 확인 ---
# (ix,iy) 키로 저장된 closed_set: 시작(2,2) -> (3,2) -> (4,3) 으로 이어진다고 가정
demo_closed = {
    (2, 2): Node(2, 2, 0.00, -1),         # 시작 (부모 없음)
    (3, 2): Node(3, 2, 1.00, (2, 2)),
    (4, 3): Node(4, 3, 2.41, (3, 2)),
}
demo_goal = demo_closed[(4, 3)]
px, py = reconstruct_path(demo_goal, demo_closed, min_x, min_y, RESOLUTION)
print('parent 체인:  (4,3) -> (3,2) -> (2,2)')
print('역추적 경로 (시작 -> 목표 순, 좌표[m]):')
for x, y in zip(px, py):
    print(f'   ({x:.0f}, {y:.0f})')

## 11. 전체 루프 조립

지금까지의 조각(초기화 -> 한 번의 확장 -> 목표 확인 -> 역추적)을 `while` 루프로 묶습니다.
**최적성**: 모든 비용이 **0 이상** 이므로, Closed Set에 들어간 노드의 비용은 더 줄지 않습니다.
**복잡도**: O((V+E) log V).

In [ ]:
DijkstraResult = namedtuple('DijkstraResult', 'path_x path_y closed_x closed_y explore_order')

def dijkstra(sx, sy, gx, gy, ox, oy, resolution, robot_radius):
    obstacle_map, min_x, min_y, max_x, max_y, x_width, y_width = build_obstacle_map(ox, oy, resolution, robot_radius)

    start = Node(calc_grid_index(sx, min_x, resolution),
                 calc_grid_index(sy, min_y, resolution), 0.0, -1)
    goal  = Node(calc_grid_index(gx, min_x, resolution),
                 calc_grid_index(gy, min_y, resolution), 0.0, -1)

    priority_queue = [(0.0, start)]   # Open Set (MinHeap)
    closed_set = {}                   # Closed Set, 키 = (ix, iy)
    explore_order = []                # 시각화용 탐색 순서

    while priority_queue:
        _, current = heapq.heappop(priority_queue)        # (1) 비용 최소 노드를 꺼냄
        current_key = (current.x, current.y)

        if current_key in closed_set:                     # 이미 확정된 중복이면 건너뜀
            continue
        
        closed_set[current_key] = current                 # (2) Closed로 확정
        explore_order.append((calc_grid_position(current.x, min_x, resolution),
                              calc_grid_position(current.y, min_y, resolution), current.cost))
        
        if current.x == goal.x and current.y == goal.y:   # 목표 도달 -> 종료
            # ==========================================================================
            # TODO 3 (1/2): 목표 도달 시 목표 노드의 부모 기록 (이후 경로 역추적에 사용)
            goal.parent_index = current.parent_index
            goal.cost = current.cost
            # ==========================================================================
            break

        for dx, dy, cost in get_motion_model():           # (3) 8이웃을 Open에 추가
            
            # ==========================================================================
            # TODO 3 (2/2): 현재 노드(current)에서 한 칸 이동한 이웃 노드를 만들어 neighbor 에 담기 (TODO 1과 동일)
            neighbor = Node(current.x + dx, current.y + dy, current.cost + cost, current_key)
            # ==========================================================================

            if not is_valid(neighbor, obstacle_map, min_x, min_y, max_x, max_y, resolution):
                continue
            if (neighbor.x, neighbor.y) in closed_set:
                continue
            heapq.heappush(priority_queue, (neighbor.cost, neighbor))

    path_x, path_y = reconstruct_path(goal, closed_set, min_x, min_y, resolution)   # 섹션 10의 역추적
    closed_x = [calc_grid_position(n.x, min_x, resolution) for n in closed_set.values()]
    closed_y = [calc_grid_position(n.y, min_y, resolution) for n in closed_set.values()]
    return DijkstraResult(path_x, path_y, closed_x, closed_y, explore_order)

## 12. 탐색 실행

탐색 노드 수(전체 셀 대비 %)가 효율 지표입니다. 경로 길이는 A*와 비교해도 동일해야 합니다.

In [ ]:
start_time = time.time()
result = dijkstra(sx, sy, gx, gy, ox, oy, RESOLUTION, ROBOT_RADIUS)
elapsed_ms = (time.time() - start_time) * 1000

path_x, path_y = result.path_x, result.path_y
closed_x, closed_y = result.closed_x, result.closed_y
explore_order = result.explore_order

path_len = sum(math.hypot(path_x[i+1]-path_x[i], path_y[i+1]-path_y[i]) for i in range(len(path_x)-1))
print(f'탐색 노드 {len(closed_x):,}개  ({len(closed_x)/x_width/y_width*100:.0f}% of {x_width*y_width:,})')
print(f'경로 길이 {path_len:.1f} m,  탐색 시간 {elapsed_ms:.0f} ms')

## 13. 탐색 과정 애니메이션

색(plasma)이 낮은 비용(보라)부터 퍼집니다. 목표 방향을 모르므로 **방사형**으로 확산 -> Dijkstra의 비효율 원인.

In [ ]:
# 비용(plasma) 순으로 퍼지다 목표 도달 후 경로를 역추적 (utils.search_viz)
HTML(animate_search(explore_order, path_x, path_y, ox, oy, sx, sy, gx, gy,
                    title='Dijkstra').to_jshtml())

## 14. 해상도(Resolution) 비교

이제 알고리즘을 직접 돌려 봤으니, **해상도만 바꿔** 같은 Dijkstra를 실행해 그 영향을 확인합니다.
해상도가 촘촘할수록(값이 작아짐) 셀/탐색 노드/시간이 늘지만 경로는 정밀해지고, 성길수록(값이 커짐) 빠르지만 거칩니다 - 섹션 2에서 미뤄 둔 **정밀도 <-> 계산량 트레이드오프**입니다.

> [주의] 단, 해상도가 너무 낮으면(res=4m) 얇은 내부 벽이 격자에서 사라져 경로가 벽을 가로질러 버립니다.
> 아래 표에서 res=4m의 경로가 오히려 짧아 보이는 것은 '더 최적'이라서가 아니라 이 **이산화 오류(벽 통과)** 때문입니다 - 해상도는 무조건 낮다고 좋은 게 아닙니다.
> 아래 **격자 그림**에서 res=4m의 경로(빨강)가 원본 벽(검정)을 그대로 가로지르는 것을 직접 확인할 수 있습니다 — 격자가 너무 성겨 그 벽을 통행 가능한 빈칸으로 보기 때문입니다.

In [ ]:
resolutions = [1.0, 2.0, 4.0]
results = []
print(f"{'res[m]':>7}{'cells':>10}{'explored':>10}{'path[m]':>10}{'time[ms]':>10}")
print('-' * 47)

for res in resolutions:
    t0 = time.time()
    r = dijkstra(sx, sy, gx, gy, ox, oy, res, ROBOT_RADIUS)
    elapsed = (time.time() - t0) * 1000
    plen = sum(math.hypot(r.path_x[i+1]-r.path_x[i], r.path_y[i+1]-r.path_y[i])
               for i in range(len(r.path_x)-1))
    cells = (int((max_x - min_x) / res) + 1) * (int((max_y - min_y) / res) + 1)
    results.append((res, r, plen, cells))
    print(f'{res:>7.1f}{cells:>10,}{len(r.closed_x):>10,}{plen:>10.1f}{elapsed:>10.0f}')

# 해상도별 격자선 + 원본 벽 + 경로: 촘촘할수록 칸이 작고, 성기면 벽을 못 잡아 경로가 벽을 가로지른다
fig, axes = plt.subplots(1, 3, figsize=(15, 5.2))
for ax, (res, r, plen, cells) in zip(axes, results):
    xw = int((max_x - min_x) / res) + 1
    yw = int((max_y - min_y) / res) + 1

    for x in np.arange(min_x, max_x + res, res):
        ax.axvline(x, color='lightgray', lw=0.5)
    for y in np.arange(min_y, max_y + res, res):
        ax.axhline(y, color='lightgray', lw=0.5)

    ax.plot(ox, oy, '.k', ms=1.5, alpha=0.5)              # 원본 벽
    ax.plot(r.path_x, r.path_y, '-r', lw=2, zorder=4)     # Dijkstra 경로
    ax.plot(sx, sy, 'og', ms=9); ax.plot(gx, gy, 'xb', ms=9, mew=2)
    used = '   <-- USED' if res == RESOLUTION else ''
    ax.set_aspect('equal'); ax.grid(False)
    ax.set_title(f'resolution = {res} m{used}\n{xw} x {yw} = {cells:,} cells, path {plen:.0f} m', fontsize=10)
    
plt.suptitle('Resolution vs Grid Granularity  (finer = more cells = precise but slow)', fontsize=12)
plt.tight_layout(rect=[0, 0, 1, 0.90]); plt.show()

# 탐색 노드 수 막대
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar([f'{res} m' for res, *_ in results], [len(r.closed_x) for _, r, *_ in results],
       color=['#4C72B0', '#55A868', '#C44E52'])
for i, (_, r, *_) in enumerate(results):
    ax.text(i, len(r.closed_x), f'{len(r.closed_x):,}', ha='center', va='bottom', fontsize=9)
ax.set_ylabel('explored nodes'); ax.set_title('Explored nodes vs resolution (finer = more)')
plt.tight_layout(); plt.show()

## 15. 결과 요약

### Dijkstra 특징

| 항목 | 내용 |
|------|------|
| 최적성 | 보장 (모든 비용 >= 0) |
| 완전성 | 보장 |
| 방향성 | 없음 -> 전방위 방사형 |
| 한계 | 목표와 무관한 노드도 탐색 |

이 표의 **'최적성 보장'** 과 **'목표 무관 탐색'** 이 바로 **최적성 <-> 계산시간 트레이드오프**입니다.
섹션 12에서 Dijkstra는 전체 격자의 약 **83%(약 1,071개)** 를 탐색했습니다 - 목표 방향을 모른 채 사방으로 퍼지기 때문입니다.

-> **Step 03 A***: 우선순위를 `g(n)` -> `g(n)+h(n)`로 바꿔 목표 방향에 집중합니다.
같은 맵에서 **탐색 노드가 얼마나 줄어드는지**, 그리고 **경로 길이는 그대로 최적(약 127.2 m)** 인지 직접 비교하게 됩니다.

In [ ]:
fig = plot_search_result(closed_x, closed_y, path_x, path_y,
                         ox, oy, sx, sy, gx, gy, title=f'Dijkstra ({len(closed_x):,} nodes)')
plt.tight_layout(); plt.show()